In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import linear_kernel

import pickle


In [2]:
movies = pd.read_csv("../data/processed/movies_cleaned.csv")

In [3]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

In [4]:
movies['tags'] = movies['tags'].fillna('')

In [5]:
C = movies['vote_average'].mean()
m = movies['vote_count'].quantile(0.90)

def weighted_rating(row):
    v = row['vote_count']
    R = row['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

movies['weighted_rating'] = movies.apply(weighted_rating, axis=1)

In [6]:
movies = movies.sample(20000, random_state=42).reset_index(drop=True)


In [7]:
tfidf_matrix = tfidf.fit_transform(movies['tags'])

In [8]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 552649 stored elements and shape (20000, 5000)>

In [9]:
pickle.dump(tfidf_matrix, open('../models/tfidf_matrix.pkl', 'wb'))

In [10]:
movies.nlargest(10, 'vote_count')[['title', 'vote_count']]

,title,vote_count
6685,Avatar,12114.0
17660,The Avengers,12000.0
427,Deadpool,11444.0
3839,Guardians of the Galaxy,10014.0
11200,The Dark Knight Rises,9263.0
18925,Pulp Fiction,8670.0
2707,The Hobbit: An Unexpected Journey,8427.0
8751,The Shawshank Redemption,8358.0
14216,The Lord of the Rings: The Return of the King,8226.0
19295,Forrest Gump,8147.0


In [11]:
pickle.dump(tfidf_matrix, open('../models/tfidf_matrix.pkl', 'wb'))
movies.to_csv('../data/processed/movies_final.csv', index=False)